# Notebook for training different neural architectures 

In [1]:
import numpy as np
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
from torch.utils.data import DataLoader
import torchvision.datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torch.nn.functional as F
from torchvision.models import resnet18
from torch import Tensor



import csv
from argparse import Namespace
from typing import Dict, List, Optional

'''
import tensorflow as tf
from tensorflow.keras.models import Model, load_model, save_model, Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Dropout, GlobalAveragePooling2D, Dense, Input, Conv2D, MaxPooling2D, Flatten, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
from time import time

'''

'\nimport tensorflow as tf\nfrom tensorflow.keras.models import Model, load_model, save_model, Sequential\nfrom tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Dropout, GlobalAveragePooling2D, Dense, Input, Conv2D, MaxPooling2D, Flatten, Layer\nfrom tensorflow.keras.optimizers import Adam\nfrom tensorflow.keras.utils import to_categorical\n\nimport matplotlib.pyplot as plt\nfrom time import time\n\n'

### Load MNIST dataset

In [2]:
# Keras
(x_train, y_train_keras), (x_test, y_test_keras) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
x_train = np.reshape(x_train, x_train.shape + (1,))
x_test = np.reshape(x_test, x_test.shape + (1,))
#y_train_keras = to_categorical(y_train)
#y_test_keras = to_categorical(y_test)
xmin, xmax = -1, 1
x_train_keras = ((x_train - x_train.min()) / (x_train.max() - x_train.min())) * (xmax - xmin) + xmin
x_test_keras = ((x_test - x_test.min()) / (x_test.max() - x_test.min())) * (xmax - xmin) + xmin

NameError: name 'tf' is not defined

In [3]:
# Torch
transform = transforms.Compose(
		[transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])

test_set_torch = torchvision.datasets.MNIST(root='./sample_data', download=True, train=False, transform=transform)
train_set_torch = torchvision.datasets.MNIST(root='./sample_data', download=True, train=True, transform=transform)
train_loader_torch = DataLoader(train_set_torch, batch_size=8, shuffle=True)
test_loader_torch = DataLoader(test_set_torch, batch_size=8, shuffle=False)

In [4]:
for i, (images, labels) in enumerate(test_loader_torch):
    if i <= 1:
        print("image:", images)
        print("label:", labels)
    else:
        pass
    

image: tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        ...,


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
         

### train torch version of CNN 6 layers 

In [22]:

class CNN_L6_128(nn.Module):
	
	def __init__(self):
		super(CNN_L6_128, self).__init__()
		self.main = nn.Sequential(
			
			# input is Z, going into a convolution
			nn.Conv2d(1, 8, kernel_size=5, stride=1, padding=2),
			nn.BatchNorm2d(8),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),
			
			nn.Conv2d(8, 16, kernel_size=5, stride=2, padding=2),
			nn.BatchNorm2d(16),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),

			nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
			nn.BatchNorm2d(32),
			nn.ReLU(True),
			nn.Dropout2d(p=0.1),

			nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2),
			nn.BatchNorm2d(64),
			nn.ReLU(True),
			nn.Dropout2d(p=0.2),

			nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
			nn.BatchNorm2d(128),
			nn.ReLU(True)
		)
			
		self.classifier = nn.Sequential(
			nn.Linear(128, 10),
		)
		
	def forward(self, x):   
		x = self.main(x)
		x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)  # GAP Layer
		logits = self.classifier(x)
		return logits, x
	

In [23]:
CUDA = True and torch.cuda.is_available()
device = torch.device("cuda:0" if CUDA else "cpu")
model = CNN_L6_128().to(device)

In [24]:
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

# 6. Training the model
num_epochs = 10

In [25]:
len(train_loader_torch)

7500

In [ ]:
for epoch in range(num_epochs):
  running_loss = 0.0
  correct_samples = 0
  sum_samples = 0

  for i, (images, labels) in enumerate(train_loader_torch):
    images, labels = images.to(device), labels.to(device)
    #print(model(images))
    #images = images.unsqueeze(0)
    #print(model(images))

    logits, _ = model(images)

    pred_softmax = F.softmax(logits, dim=1)

    loss = criterion(pred_softmax, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    predicted = torch.argmax(pred_softmax, 1)
    
    correct_samples += (predicted == labels).sum().item()
    sum_samples += labels.size(0)

  accuracy = correct_samples / sum_samples
  print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_set_torch):.4f}, Accuracy: {accuracy}')

Epoch [1/10], Loss: 0.2010, Accuracy: 0.8701333333333333
Epoch [2/10], Loss: 0.1883, Accuracy: 0.9596833333333333
Epoch [3/10], Loss: 0.1869, Accuracy: 0.9690166666666666
Epoch [4/10], Loss: 0.1863, Accuracy: 0.9728833333333333
Epoch [5/10], Loss: 0.1859, Accuracy: 0.9752333333333333
Epoch [6/10], Loss: 0.1855, Accuracy: 0.9789333333333333
Epoch [7/10], Loss: 0.1853, Accuracy: 0.9798666666666667


In [20]:
model.eval()

# Initialize variables to track correct predictions and total samples
correct = 0
total = 0

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_set_torch:
        # Move images and labels to the appropriate device
        #images, labels = images.to(device), labels.to(device)
        
        # Forward pass to get the model's predictions
        images = images.unsqueeze(0)
        outputs, _ = model(images)
        
        # Get predicted class by choosing the class with the highest score
        _, predicted = torch.max(outputs, 1)
        labels = torch.tensor([labels])
        
        # Update total count and correct count
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 99.25%


In [ ]:
# save torch model
torch.save(model.state_dict(), "mnist_cnn_6_128_norm01.pt")

In [11]:
def Cnn_6_128():
    input_layer = Input(shape=(28, 28, 1))  # Assuming input shape is (28, 28, 1)
    
    # Layer 1
    x = Conv2D(8, kernel_size=5, strides=1, padding='same')(input_layer)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.1)(x)
    
    # Layer 2
    x = Conv2D(16, kernel_size=5, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.1)(x)
    
    # Layer 3
    x = Conv2D(32, kernel_size=5, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.1)(x)
    
    # Layer 4
    x = Conv2D(64, kernel_size=5, strides=2, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.2)(x)
    
    # Layer 5
    x = Conv2D(128, kernel_size=3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Global Average Pooling
    x = GlobalAveragePooling2D()(x)
    
    # Dense (fully connected) layer for classification
    output_layer = Dense(10, activation='softmax')(x)  # 10 classes, softmax for classification
    
    # Define the model
    model = Model(inputs=input_layer, outputs=output_layer)
    
    return model

# Build and summarize the model
model = Cnn_6_128()
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_5 (Conv2D)           (None, 28, 28, 8)         208       
                                                                 
 batch_normalization_5 (Batc  (None, 28, 28, 8)        32        
 hNormalization)                                                 
                                                                 
 activation_5 (Activation)   (None, 28, 28, 8)         0         
                                                                 
 dropout_4 (Dropout)         (None, 28, 28, 8)         0         
                                                                 
 conv2d_6 (Conv2D)           (None, 14, 14, 16)        3216      
                                                           

In [12]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # Use sparse_categorical_crossentropy for integer labels
    metrics=['accuracy'])

In [13]:
# Set training parameters
batch_size = 8
epochs = 10

# Train the model
history = model.fit(
    x_train_keras, y_train_keras,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test_keras, y_test_keras)
)

Epoch 1/10
7500/7500 [==============================] - 89s 12ms/step - loss: 0.1776 - accuracy: 0.9482 - val_loss: 0.0602 - val_accuracy: 0.9803
Epoch 2/10
7500/7500 [==============================] - 82s 11ms/step - loss: 0.0700 - accuracy: 0.9786 - val_loss: 0.0274 - val_accuracy: 0.9907
Epoch 3/10
7500/7500 [==============================] - 85s 11ms/step - loss: 0.0546 - accuracy: 0.9837 - val_loss: 0.0302 - val_accuracy: 0.9895
Epoch 4/10
7500/7500 [==============================] - 83s 11ms/step - loss: 0.0464 - accuracy: 0.9858 - val_loss: 0.0266 - val_accuracy: 0.9909
Epoch 5/10
7500/7500 [==============================] - 82s 11ms/step - loss: 0.0416 - accuracy: 0.9872 - val_loss: 0.0313 - val_accuracy: 0.9897
Epoch 6/10
7500/7500 [==============================] - 81s 11ms/step - loss: 0.0374 - accuracy: 0.9886 - val_loss: 0.0212 - val_accuracy: 0.9935
Epoch 7/10
7500/7500 [==============================] - 82s 11ms/step - loss: 0.0322 - accuracy: 0.9899 - val_loss: 0.0203 -

In [14]:
test_loss, test_accuracy = model.evaluate(x_test_keras, y_test_keras, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.0198, Test Accuracy: 0.9934


In [15]:
model.save('mnist_cnn_6_128.h5')

## Train LeNet-5

### Torch

In [5]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2) # (28x28 -> 28x28)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0) # (14x14 -> 10x10)

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 16 channels, 5x5 feature maps
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
        
    def forward(self, x):
        # First convolutional layer + activation + pooling
        x = F.relu(self.conv1(x))       # Output: (6, 28, 28)
        x = F.max_pool2d(x, kernel_size=2, stride=2) # Output: (6, 14, 14)
        
        # Second convolutional layer + activation + pooling
        x = F.relu(self.conv2(x))       # Output: (16, 10, 10)
        x = F.max_pool2d(x, kernel_size=2, stride=2) # Output: (16, 5, 5)
        
        # Flatten the tensor for the fully connected layers
        x = x.view(-1, 16 * 5 * 5)     # Output: (batch_size, 16*5*5)
        
        # Fully connected layers + activations
        x = F.relu(self.fc1(x))        # Output: (batch_size, 120)
        x = F.relu(self.fc2(x))        # Output: (batch_size, 84)
        x = self.fc3(x)                # Output: (batch_size, num_classes)
        
        return x

In [6]:
CUDA = True and torch.cuda.is_available()
device = torch.device("cuda:0" if CUDA else "cpu")
model = LeNet5().to(device)

In [7]:
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

# 6. Training the model
num_epochs = 10

In [10]:
for epoch in range(num_epochs):
  running_loss = 0.0
  correct_samples = 0
  sum_samples = 0

  for i, (images, labels) in enumerate(train_loader_torch):
    images, labels = images.to(device), labels.to(device)
    #print(model(images))
    #images = images.unsqueeze(0)
    #print(model(images))

    logits = model(images)

    pred_softmax = F.softmax(logits, dim=1)

    loss = criterion(pred_softmax, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    predicted = torch.argmax(pred_softmax, 1)
    
    correct_samples += (predicted == labels).sum().item()
    sum_samples += labels.size(0)

  accuracy = correct_samples / sum_samples
  print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_set_torch):.4f}, Accuracy: {accuracy}')

Epoch [1/10], Loss: 0.1926, Accuracy: 0.92125
Epoch [2/10], Loss: 0.1869, Accuracy: 0.9660833333333333
Epoch [3/10], Loss: 0.1865, Accuracy: 0.9692666666666667
Epoch [4/10], Loss: 0.1864, Accuracy: 0.9696333333333333
Epoch [5/10], Loss: 0.1865, Accuracy: 0.9694
Epoch [6/10], Loss: 0.1866, Accuracy: 0.9684333333333334
Epoch [7/10], Loss: 0.1869, Accuracy: 0.9661
Epoch [8/10], Loss: 0.1867, Accuracy: 0.9673333333333334
Epoch [9/10], Loss: 0.1865, Accuracy: 0.9691
Epoch [10/10], Loss: 0.1870, Accuracy: 0.9649666666666666


In [12]:
model.eval()

# Initialize variables to track correct predictions and total samples
correct = 0
total = 0

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_set_torch:
        # Move images and labels to the appropriate device
        #images, labels = images.to(device), labels.to(device)
        
        # Forward pass to get the model's predictions
        images = images.unsqueeze(0)
        outputs = model(images)
        
        # Get predicted class by choosing the class with the highest score
        _, predicted = torch.max(outputs, 1)
        labels = torch.tensor([labels])
        
        # Update total count and correct count
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 97.26%


In [13]:
# save torch model
torch.save(model.state_dict(), "mnist_lenet5.pt")

### Keras

In [16]:
def lenet5(input_shape=(28, 28, 1), num_classes=10):
    model = Sequential()
    
    # First convolutional layer
    model.add(Conv2D(6, kernel_size=(5, 5), strides=(1, 1), padding='same', input_shape=input_shape))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    # Second convolutional layer
    model.add(Conv2D(16, kernel_size=(5, 5), strides=(1, 1), padding='valid'))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    
    # Flatten the output
    model.add(Flatten())
    
    # Fully connected layers
    model.add(Dense(120, activation='relu'))
    model.add(Dense(84, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Output layer
    
    return model

# Compile and build the model
model = lenet5()
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 6)         156       
                                                                 
 activation (Activation)     (None, 28, 28, 6)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 14, 14, 6)        0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 10, 10, 16)        2416      
                                                                 
 activation_1 (Activation)   (None, 10, 10, 16)        0         
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 5, 5, 16)         0         
 2D)                                                    

In [17]:
# Set training parameters
batch_size = 8
epochs = 10

# Train the model
history = model.fit(
    x_train_keras, y_train_keras,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test_keras, y_test_keras))

Epoch 1/10
7500/7500 [==============================] - 27s 4ms/step - loss: 0.1288 - accuracy: 0.9609 - val_loss: 0.0647 - val_accuracy: 0.9802
Epoch 2/10
7500/7500 [==============================] - 26s 3ms/step - loss: 0.0532 - accuracy: 0.9842 - val_loss: 0.0427 - val_accuracy: 0.9870
Epoch 3/10
7500/7500 [==============================] - 25s 3ms/step - loss: 0.0412 - accuracy: 0.9875 - val_loss: 0.0461 - val_accuracy: 0.9857
Epoch 4/10
7500/7500 [==============================] - 26s 3ms/step - loss: 0.0339 - accuracy: 0.9899 - val_loss: 0.0452 - val_accuracy: 0.9874
Epoch 5/10
7500/7500 [==============================] - 26s 3ms/step - loss: 0.0286 - accuracy: 0.9911 - val_loss: 0.0443 - val_accuracy: 0.9872
Epoch 6/10
7500/7500 [==============================] - 26s 3ms/step - loss: 0.0253 - accuracy: 0.9927 - val_loss: 0.0481 - val_accuracy: 0.9882
Epoch 7/10
7500/7500 [==============================] - 26s 4ms/step - loss: 0.0246 - accuracy: 0.9928 - val_loss: 0.0423 - val_ac

In [18]:
test_loss, test_accuracy = model.evaluate(x_test_keras, y_test_keras, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.0630, Test Accuracy: 0.9882


In [61]:
model.save('mnist_lenet5.h5')

NotImplementedError: Saving the model to HDF5 format requires the model to be a Functional model or a Sequential model. It does not work for subclassed models, because such models are defined via the body of a Python method, which isn't safely serializable. Consider saving to the Tensorflow SavedModel format (by setting save_format="tf") or using `save_weights`.

## Train Resnet-8 (please double check before you officially run)

### Torch

In [67]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 16

        # Initial convolution (smaller for MNIST)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual layers
        self.layer1 = self._make_layer(block, 16, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 32, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 64, layers[2], stride=2)

        # Adaptive average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x




In [68]:
CUDA = True and torch.cuda.is_available()
device = torch.device("cuda:0" if CUDA else "cpu")
model = ResNet(BasicBlock, [1, 1, 1], num_classes=10).to(device)

In [69]:
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

# 6. Training the model
num_epochs = 10

In [70]:
for epoch in range(num_epochs):
  running_loss = 0.0
  correct_samples = 0
  sum_samples = 0

  for i, (images, labels) in enumerate(train_loader_torch):
    images, labels = images.to(device), labels.to(device)
    #print(model(images))
    #images = images.unsqueeze(0)
    #print(model(images))

    logits = model(images)

    pred_softmax = F.softmax(logits, dim=1)

    loss = criterion(pred_softmax, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    predicted = torch.argmax(pred_softmax, 1)
    
    correct_samples += (predicted == labels).sum().item()
    sum_samples += labels.size(0)

  accuracy = correct_samples / sum_samples
  print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_set_torch):.4f}, Accuracy: {accuracy}')

Epoch [1/10], Loss: 0.1929, Accuracy: 0.9345166666666667
Epoch [2/10], Loss: 0.1856, Accuracy: 0.9793166666666666
Epoch [3/10], Loss: 0.1849, Accuracy: 0.9835166666666667
Epoch [4/10], Loss: 0.1846, Accuracy: 0.9859833333333333
Epoch [5/10], Loss: 0.1844, Accuracy: 0.9876666666666667
Epoch [6/10], Loss: 0.1842, Accuracy: 0.9888833333333333
Epoch [7/10], Loss: 0.1841, Accuracy: 0.9895833333333334
Epoch [8/10], Loss: 0.1839, Accuracy: 0.9904666666666667
Epoch [9/10], Loss: 0.1838, Accuracy: 0.9909333333333333
Epoch [10/10], Loss: 0.1837, Accuracy: 0.9919666666666667


In [71]:
model.eval()

# Initialize variables to track correct predictions and total samples
correct = 0
total = 0

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_set_torch:
        # Move images and labels to the appropriate device
        #images, labels = images.to(device), labels.to(device)
        
        # Forward pass to get the model's predictions
        images = images.unsqueeze(0)
        outputs = model(images)
        
        # Get predicted class by choosing the class with the highest score
        _, predicted = torch.max(outputs, 1)
        labels = torch.tensor([labels])
        
        # Update total count and correct count
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 99.18%


In [72]:
# save torch model
torch.save(model.state_dict(), "mnist_resnet8.pt")

### Keras

In [62]:
# Basic Block
class BasicBlock(Layer):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = Conv2D(out_channels, kernel_size=3, strides=stride, padding="same", use_bias=False)
        self.bn1 = BatchNormalization()
        self.relu = Activation('relu')
        self.conv2 = Conv2D(out_channels, kernel_size=3, strides=1, padding="same", use_bias=False)
        self.bn2 = BatchNormalization()
        self.downsample = downsample

    def call(self, inputs, training=False):
        identity = inputs
        if self.downsample is not None:
            identity = self.downsample(inputs)

        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)

        x += identity
        x = self.relu(x)

        return x


# ResNet
class ResNet(Model):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 16

        # Initial convolution
        self.conv1 = Conv2D(16, kernel_size=3, strides=1, padding="same", use_bias=False)
        self.bn1 = BatchNormalization()
        self.relu = Activation('relu')

        # Residual layers
        self.layer1 = self._make_layer(block, 16, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 32, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 64, layers[2], stride=2)

        # Classification head
        self.avgpool = GlobalAveragePooling2D()
        self.fc = Dense(num_classes, activation='softmax')

    def _make_layer(self, block, out_channels, blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = tf.keras.Sequential([
                Conv2D(out_channels, kernel_size=1, strides=stride, use_bias=False),
                BatchNormalization()
            ])

        layers_list = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers_list.append(block(self.in_channels, out_channels))

        return tf.keras.Sequential(layers_list)

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = self.relu(x)

        x = self.layer1(x, training=training)
        x = self.layer2(x, training=training)
        x = self.layer3(x, training=training)

        x = self.avgpool(x)
        x = self.fc(x)

        return x


# Instantiate the Model
model = ResNet(BasicBlock, [1, 1, 1])  # Using a shallow ResNet for MNIST
model.build(input_shape=(None, 28, 28, 1))  # MNIST input size
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])
model.summary()


Model: "res_net_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_47 (Conv2D)          multiple                  144       
                                                                 
 batch_normalization_45 (Bat  multiple                 64        
 chNormalization)                                                
                                                                 
 activation_22 (Activation)  multiple                  0         
                                                                 
 sequential_26 (Sequential)  (None, 28, 28, 16)        4736      
                                                                 
 sequential_28 (Sequential)  (None, 14, 14, 32)        14720     
                                                                 
 sequential_30 (Sequential)  (None, 7, 7, 64)          58112     
                                                         

In [63]:
# Set training parameters
batch_size = 8
epochs = 10

# Train the model
history = model.fit(
    x_train_keras, y_train_keras,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test_keras, y_test_keras))

Epoch 1/10
7500/7500 [==============================] - 103s 14ms/step - loss: 0.1727 - accuracy: 0.9525 - val_loss: 0.0955 - val_accuracy: 0.9710
Epoch 2/10
7500/7500 [==============================] - 102s 14ms/step - loss: 0.0578 - accuracy: 0.9824 - val_loss: 0.0946 - val_accuracy: 0.9708
Epoch 3/10
7500/7500 [==============================] - 114s 15ms/step - loss: 0.0424 - accuracy: 0.9874 - val_loss: 0.2877 - val_accuracy: 0.9009
Epoch 4/10
7500/7500 [==============================] - 102s 14ms/step - loss: 0.0346 - accuracy: 0.9891 - val_loss: 0.0403 - val_accuracy: 0.9867
Epoch 5/10
7500/7500 [==============================] - 102s 14ms/step - loss: 0.0288 - accuracy: 0.9913 - val_loss: 0.0300 - val_accuracy: 0.9902
Epoch 6/10
7500/7500 [==============================] - 101s 13ms/step - loss: 0.0255 - accuracy: 0.9922 - val_loss: 0.0224 - val_accuracy: 0.9919
Epoch 7/10
7500/7500 [==============================] - 102s 14ms/step - loss: 0.0216 - accuracy: 0.9932 - val_loss: 0

In [64]:
test_loss, test_accuracy = model.evaluate(x_test_keras, y_test_keras, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.0219, Test Accuracy: 0.9924


In [66]:
model.save_weights("mnist_resnet8_weights.h5")

## Train CNN (from Alibi)

### Torch

In [93]:
class AlibiCNN(nn.Module):
    def __init__(self):
        super(AlibiCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=2, padding=1)  
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.dropout1 = nn.Dropout(0.3)

   
        self.fc1 = nn.Linear(64 * 14 * 14, 256) 
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = self.dropout1(x)

        x = torch.flatten(x, 1) 

        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)  # Final layer without activation (will apply softmax in loss function)

        return x

In [94]:
CUDA = True and torch.cuda.is_available()
device = torch.device("cuda:0" if CUDA else "cpu")
model = AlibiCNN().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

num_epochs = 3

In [96]:
train_loader_torch = DataLoader(train_set_torch, batch_size=32, shuffle=True)
test_loader_torch = DataLoader(test_set_torch, batch_size=32, shuffle=False)

In [97]:
for epoch in range(num_epochs):
  running_loss = 0.0
  correct_samples = 0
  sum_samples = 0

  for i, (images, labels) in enumerate(train_loader_torch):
    images, labels = images.to(device), labels.to(device)
    #print(model(images))
    #images = images.unsqueeze(0)
    #print(model(images))

    logits = model(images)

    pred_softmax = F.softmax(logits, dim=1)

    loss = criterion(pred_softmax, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    predicted = torch.argmax(pred_softmax, 1)
    
    correct_samples += (predicted == labels).sum().item()
    sum_samples += labels.size(0)

  accuracy = correct_samples / sum_samples
  print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_set_torch):.4f}, Accuracy: {accuracy}')

Epoch [1/3], Loss: 0.0510, Accuracy: 0.8357666666666667
Epoch [2/3], Loss: 0.0477, Accuracy: 0.9356333333333333
Epoch [3/3], Loss: 0.0473, Accuracy: 0.9478166666666666


In [98]:
model.eval()

# Initialize variables to track correct predictions and total samples
correct = 0
total = 0

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_set_torch:
        # Move images and labels to the appropriate device
        #images, labels = images.to(device), labels.to(device)
        
        # Forward pass to get the model's predictions
        images = images.unsqueeze(0)
        outputs = model(images)
        
        # Get predicted class by choosing the class with the highest score
        _, predicted = torch.max(outputs, 1)
        labels = torch.tensor([labels])
        
        # Update total count and correct count
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 97.20%


In [99]:
# save torch model
torch.save(model.state_dict(), "mnist_alibi_cnn.pt")

### Keras

In [87]:
y_train = to_categorical(y_train_keras)
y_test = to_categorical(y_test_keras)

In [88]:
def alibi_cnn():
    x_in = Input(shape=(28,28,1))
    x = Conv2D(filters=64, kernel_size=2, padding='same', activation='relu')(x_in)
    x = MaxPooling2D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x_out = Dense(10, activation='softmax')(x)
    
    cnn = Model(inputs=x_in, outputs=x_out)
    cnn.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


    return cnn


In [89]:
cnn = alibi_cnn()
cnn.summary()

Model: "model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_62 (Conv2D)          (None, 28, 28, 64)        320       
                                                                 
 max_pooling2d_8 (MaxPooling  (None, 14, 14, 64)       0         
 2D)                                                             
                                                                 
 dropout_12 (Dropout)        (None, 14, 14, 64)        0         
                                                                 
 flatten_7 (Flatten)         (None, 12544)             0         
                                                                 
 dense_20 (Dense)            (None, 256)               3211520   
                                                           

In [90]:
cnn.fit(x_train_keras, y_train, batch_size=32, epochs=3, verbose=0)

In [91]:
score = cnn.evaluate(x_test_keras, y_test, verbose=0)
print('Test accuracy: ', score[1])

Test accuracy:  0.9818999767303467


In [92]:
cnn.save('mnist_alibi_cnn.h5', save_format='h5')

## Train MLP (from Schut et al)

### Torch

In [16]:
class MLP_schut(nn.Module):
    """A single multi-layer perceptron for classification."""

    def __init__(self, n_hidden=80, input_flat_size=28*28, n_classes=10):
        super().__init__()
        self._net = nn.Sequential(  #
            nn.modules.flatten.Flatten(),  #
            nn.Linear(in_features=input_flat_size, out_features=n_hidden),  #
            nn.ReLU(),  #
            nn.BatchNorm1d(num_features=n_hidden),  #
            nn.Linear(in_features=n_hidden, out_features=n_hidden),  #
            nn.ReLU(),  #
            nn.BatchNorm1d(num_features=n_hidden),  #
            
        )
        self.classifier = nn.Linear(in_features=n_hidden, out_features=n_classes)

    def forward(self, x: torch.Tensor):
        x = self._net(x)
        logits = self.classifier(x)
        #probs = F.softmax(x, dim=-1)
        return logits

In [17]:
CUDA = True and torch.cuda.is_available()
device = torch.device("cuda:0" if CUDA else "cpu")
model = MLP_schut().to(device)

In [18]:
criterion = nn.CrossEntropyLoss()  # Loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

num_epochs = 20

In [19]:
train_loader_torch = DataLoader(train_set_torch, batch_size=8, shuffle=True)
test_loader_torch = DataLoader(test_set_torch, batch_size=8, shuffle=False)

In [20]:
for epoch in range(num_epochs):
  running_loss = 0.0
  correct_samples = 0
  sum_samples = 0

  for i, (images, labels) in enumerate(train_loader_torch):
    images, labels = images.to(device), labels.to(device)
    #print(model(images))
    #images = images.unsqueeze(0)
    #print(model(images))

    logits = model(images)

    pred_softmax = F.softmax(logits, dim=1)

    loss = criterion(pred_softmax, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    predicted = torch.argmax(pred_softmax, 1)
    
    correct_samples += (predicted == labels).sum().item()
    sum_samples += labels.size(0)

  accuracy = correct_samples / sum_samples
  print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_set_torch):.4f}, Accuracy: {accuracy}')

Epoch [1/20], Loss: 0.2054, Accuracy: 0.8300333333333333
Epoch [2/20], Loss: 0.2000, Accuracy: 0.8644333333333334
Epoch [3/20], Loss: 0.1991, Accuracy: 0.8705833333333334
Epoch [4/20], Loss: 0.1983, Accuracy: 0.87755
Epoch [5/20], Loss: 0.1979, Accuracy: 0.8793833333333333
Epoch [6/20], Loss: 0.1973, Accuracy: 0.8843166666666666
Epoch [7/20], Loss: 0.1966, Accuracy: 0.8898666666666667
Epoch [8/20], Loss: 0.1961, Accuracy: 0.89325
Epoch [9/20], Loss: 0.1961, Accuracy: 0.8937666666666667
Epoch [10/20], Loss: 0.1959, Accuracy: 0.8941833333333333
Epoch [11/20], Loss: 0.1955, Accuracy: 0.8978333333333334
Epoch [12/20], Loss: 0.1953, Accuracy: 0.8986833333333333
Epoch [13/20], Loss: 0.1949, Accuracy: 0.9022333333333333
Epoch [14/20], Loss: 0.1947, Accuracy: 0.9040833333333333
Epoch [15/20], Loss: 0.1947, Accuracy: 0.9037666666666667
Epoch [16/20], Loss: 0.1947, Accuracy: 0.9045166666666666
Epoch [17/20], Loss: 0.1945, Accuracy: 0.9053666666666667
Epoch [18/20], Loss: 0.1942, Accuracy: 0.9077

In [21]:
model.eval()

# Initialize variables to track correct predictions and total samples
correct = 0
total = 0

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_set_torch:
        # Move images and labels to the appropriate device
        #images, labels = images.to(device), labels.to(device)
        
        # Forward pass to get the model's predictions
        images = images.unsqueeze(0)
        outputs = model(images)
        
        # Get predicted class by choosing the class with the highest score
        _, predicted = torch.max(outputs, 1)
        labels = torch.tensor([labels])
        
        # Update total count and correct count
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 94.76%


In [22]:
torch.save(model.state_dict(), "mnist_schut_mlp.pt")

### Keras

In [135]:
def MLP_schut():
    # Input layer
    input_layer = Input(shape=(28, 28, 1))  # Shape includes channel dimension (grayscale)
    
    # Flatten layer
    x = Flatten()(input_layer)
    
    # Fully connected layers with BatchNormalization and ReLU activation
    x = Dense(80)(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    
    x = Dense(80)(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    
    
    # Softmax for probabilities
    probs = Dense(10, activation='softmax')(x)  # 10 classes, softmax for classification
    
    # Create the model
    model = Model(inputs=input_layer, outputs=probs)
    
    return model

# Build and summarize the model
model = MLP_schut()
model.summary()

Model: "model_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 flatten_11 (Flatten)        (None, 784)               0         
                                                                 
 dense_22 (Dense)            (None, 80)                62800     
                                                                 
 activation_26 (Activation)  (None, 80)                0         
                                                                 
 batch_normalization_54 (Bat  (None, 80)               320       
 chNormalization)                                                
                                                                 
 dense_23 (Dense)            (None, 80)                6480      
                                                           

In [136]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # Use sparse_categorical_crossentropy for integer labels
    metrics=['accuracy'])

In [137]:
# Set training parameters
batch_size = 8
epochs = 20

# Train the model
history = model.fit(
    x_train_keras, y_train_keras,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test_keras, y_test_keras)
)

Epoch 1/20
7500/7500 [==============================] - 10s 1ms/step - loss: 0.4704 - accuracy: 0.8547 - val_loss: 0.2647 - val_accuracy: 0.9164
Epoch 2/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.3436 - accuracy: 0.8949 - val_loss: 0.2825 - val_accuracy: 0.9142
Epoch 3/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.3213 - accuracy: 0.9014 - val_loss: 0.1899 - val_accuracy: 0.9472
Epoch 4/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.3037 - accuracy: 0.9063 - val_loss: 0.1593 - val_accuracy: 0.9528
Epoch 5/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.2826 - accuracy: 0.9126 - val_loss: 0.1319 - val_accuracy: 0.9595
Epoch 6/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.2754 - accuracy: 0.9157 - val_loss: 0.1291 - val_accuracy: 0.9617
Epoch 7/20
7500/7500 [==============================] - 9s 1ms/step - loss: 0.2670 - accuracy: 0.9194 - val_loss: 0.1200 - val_accuracy

In [138]:
test_loss, test_accuracy = model.evaluate(x_test_keras, y_test_keras, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.1103, Test Accuracy: 0.9659


In [139]:
model.save('mnist_schut.h5')